In [1]:
!pip install transformers torch

In [2]:
!git clone https://github.com/harry0703/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo
!ls /content/MoneyPrinterTurbo

Cloning into '/content/MoneyPrinterTurbo'...
remote: Enumerating objects: 2210, done.
remote: Counting objects: 100% (1394/1394), done.
remote: Compressing objects: 100% (379/379), done.
remote: Total 2210 (delta 1024), reused 1015 (delta 1015), pack-reused 816 (from 1)
Receiving objects: 100% (2210/2210), 519.61 MiB | 23.84 MiB/s, done.
Resolving deltas: 100% (1272/1272), done.
Updating files: 100% (113/113), done.
app		     Dockerfile  main.py       requirements.txt  webui
config.example.toml  docs	 README-en.md  resource		 webui.bat
docker-compose.yml   LICENSE	 README.md     test		 webui.sh


In [3]:
# main.py
import re
from pathlib import Path
from transformers import pipeline
import torch
import textwrap

# Raiz do projeto
PROJECT_ROOT = Path("/content/MoneyPrinterTurbo")

# extensões relevantes
EXTS = [".md", ".py", ".yml", ".yaml", ".toml", ".txt"]


def read_file(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        print(f"[AVISO] Erro lendo {path}: {e}")
        return ""

def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"!\[.*?\]\(.*?\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"[#*>\-`~]+", " ", text)
    text = re.sub(r"\|.+\|\n(\|[-:\s]+\|\n)?(?:\|.*\|\n)+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()


def collect_text(root: Path):
    texts = []
    for file in root.rglob("*"):
        if file.suffix.lower() in EXTS and file.is_file():
            content = read_file(file)
            if len(content) > 0:
                texts.append(content)
    return texts


def chunk_text(texts, chunk_size=8000):
    combined = " ".join(texts)
    for i in range(0, len(combined), chunk_size):
        yield combined[i:i+chunk_size]

def main():
    device = 0 if torch.cuda.is_available() else -1
    print(f"⚙️  Usando {'GPU' if device==0 else 'CPU'}")

    texts = collect_text(PROJECT_ROOT)
    total_chars = sum(len(t) for t in texts)
    print(f"📚 Total de arquivos lidos: {len(texts)} — {total_chars:,} caracteres")

    gen = pipeline("text2text-generation", model="google/flan-t5-xl", device=device)
    partial_summaries = []

    # Gera resumo de cada parte do código
    for i, chunk in enumerate(chunk_text(texts), start=1):
        prompt = f"Summarize this project section focusing on architecture and components:\n\n{chunk}"
        out = gen(prompt, max_new_tokens=200, truncation=True, do_sample=False)
        summary = out[0]['generated_text']
        partial_summaries.append(summary)
        print(f"✅ Resumo parcial {i} gerado")

    # Combina todos os resumos parciais em um resumo final
    combined_summary = " ".join(partial_summaries)
    print("\n🧠 Resumo técnico final:\n")
    print(textwrap.fill(combined_summary, 100))

    # Classificação interna (estrutura de software)
    print("\n🔍 Classificando arquitetura interna...")
    classifier_internal = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
    labels_internal = ["MVC", "Layered", "Monolithic", "Hexagonal", "Clean Architecture"]
    cls_internal = classifier_internal(combined_summary, candidate_labels=labels_internal)
    print("Scores (interna):", {lab: round(score, 4) for lab, score in zip(cls_internal["labels"], cls_internal["scores"])})
    print("🏗️  Arquitetura interna:", cls_internal["labels"][0])

    # Classificação topológica (estrutura de implantação)
    print("\n🌐 Classificando arquitetura de sistema...")
    classifier_topo = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
    labels_topo = ["Client-Server", "Microservices", "Distributed", "Single Server (Monolithic)"]
    cls_topo = classifier_topo(combined_summary, candidate_labels=labels_topo)
    print("Scores (topologia):", {lab: round(score, 4) for lab, score in zip(cls_topo["labels"], cls_topo["scores"])})
    print("🖥️  Arquitetura de sistema:", cls_topo["labels"][0])

if __name__ == "__main__":
    main()

⚙️  Usando GPU
📚 Total de arquivos lidos: 42 — 244,536 caracteres


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Scores (interna): {'Layered': 0.2766, 'Hexagonal': 0.1854, 'Clean Architecture': 0.1852, 'MVC': 0.1788, 'Monolithic': 0.1741}
🏗️  Arquitetura interna: Layered

🌐 Classificando arquitetura de sistema...


Device set to use cuda:0


Scores (topologia): {'Distributed': 0.2855, 'Single Server (Monolithic)': 0.2631, 'Client-Server': 0.2267, 'Microservices': 0.2248}
🖥️  Arquitetura de sistema: Distributed
